# 🔬 Diagnostic de l'alpha-bêta de P.O.W.N.

Avant le gros match contre Stockfish (notebook 08), on regarde **comment se comporte la recherche** avec le réseau P.O.W.N. :

1. **Vitesse** : secondes par coup, positions évaluées par coup et par seconde, à profondeur 1, 2, 3, 4 — sur 8 positions
   (ouverture, milieu de partie, finale). On voit aussi si la profondeur change le coup choisi.
2. **Puzzles Lichess** (comme le notebook 02, en plus petit : ~300 puzzles, de facile à très difficile) pour **POWN@ab2** (ancienne recherche, profondeur fixe),
   avec **POWN@t2** (recherche équilibrée, 2 s par coup) et **PWN@t2** (le réseau SPAWN, même recherche, même temps) : ça dit si le réseau P.O.W.N. voit mieux que SPAWN à temps égal.

## ⚙️ Avant de lancer
1. Ajoute ton **Dataset `pown-1`** (contenant `pown.onnx`) en **Input**.
2. **Internet ON**, accélérateur **None** (tout tourne sur CPU).
3. **Save & Run All (Commit)**.

L'alpha-bêta est celle de PWN (profondeur fixe, quiescence 4, table de transposition, tri des coups par prises et échecs).
Elle n'a **pas** de null-move, de réductions (LMR) ni de fenêtres d'aspiration : les chiffres de vitesse de ce notebook
sont donc ceux d'une recherche simple, un point de départ pour décider quoi améliorer.

In [ ]:
# onnxruntime-gpu remplace onnxruntime (les deux ne cohabitent pas) ; marche aussi sans GPU.
# Version fixée : les plus récentes demandent CUDA 13, Kaggle a CUDA 12. [cuda,cudnn] installe les bibliothèques CUDA 12.
!pip uninstall -y -q onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q chess zstandard "onnxruntime-gpu[cuda,cudnn]==1.24.1"

In [ ]:
%%writefile commun_echecs.py
"""Code commun aux notebooks de benchmark échecs (PAWN, PAWN big, contrôle, SPAWN/PWN).

Tout vient de pawn_app.py : même encodage 69 octets, même recherche « arbre complet »,
même alpha-bêta + quiescence. Seule différence : pas de bruit ni de coups au hasard,
on veut mesurer la force réelle.
"""
import glob
import json
import os
import shutil
import stat
import subprocess
import tarfile
import time
import urllib.request

import chess
import chess.engine
import chess.polyglot
import numpy as np
import onnxruntime as ort

DEPOT = "LiamLitle/ia-de-liam"
BRANCHE = "main"
LFS = f"https://media.githubusercontent.com/media/{DEPOT}/{BRANCHE}/"
DOSSIER_POIDS = os.environ.get("PAWN_POIDS", "/kaggle/working/poids" if os.path.isdir("/kaggle") else "poids")
MATE = 100_000

# nom -> chemin dans le dépôt
RESEAUX = {
    "PAWN": "Pawn/pawn.onnx",
    "PAWN_BIG": "PawnBig-V1/pawn_big.onnx",
    "CONTROLE": "PawnBig-controlleur/pawn_big_controle.onnx",
    "SPAWN": "PWN-soluce/pawn_soluce.onnx",
    "POWN": "Pown/pown.onnx",
}


def poids(chemin_depot):
    """cherche le fichier dans /kaggle/input (dataset ajouté au notebook), sinon le télécharge depuis GitHub LFS"""
    nom = os.path.basename(chemin_depot)
    for racine in ("/kaggle/input", DOSSIER_POIDS):
        trouves = glob.glob(os.path.join(racine, "**", nom), recursive=True)
        trouves = [t for t in trouves if os.path.getsize(t) > 1000]  # pas un pointeur LFS
        if trouves:
            return trouves[0]
    os.makedirs(DOSSIER_POIDS, exist_ok=True)
    dest = os.path.join(DOSSIER_POIDS, nom)
    print(f"téléchargement de {chemin_depot} ...")
    urllib.request.urlretrieve(LFS + chemin_depot, dest)
    if os.path.getsize(dest) < 1000:
        raise RuntimeError(f"{dest} ressemble à un pointeur LFS, pas au vrai fichier")
    return dest


GPU_ID = 0       # carte utilisée par ce processus (Kaggle « GPU T4 x2 » en a deux)
GPU_MEM_MO = 0   # plafond mémoire GPU par réseau (0 = pas de plafond) ; fixé quand plusieurs processus partagent le GPU
_CUDA_OK = None  # None = pas encore essayé, False = échec -> on reste sur CPU sans réessayer


def providers(gpu=True):
    dispo = ort.get_available_providers()
    if gpu and _CUDA_OK is not False and "CUDAExecutionProvider" in dispo:
        try:
            ort.preload_dlls()  # charge CUDA/cuDNN depuis les paquets pip nvidia-* (déjà là avec torch sur Kaggle)
        except Exception:
            pass
        # HEURISTIC : cuDNN choisit ses algos sans essayer ceux qui demandent d'énormes
        # zones de travail (sinon des blocs de 150 Mo dépassent le plafond mémoire)
        opts = {"arena_extend_strategy": "kSameAsRequested", "cudnn_conv_algo_search": "HEURISTIC",
                "cudnn_conv_use_max_workspace": "0", "device_id": GPU_ID}
        if GPU_MEM_MO:
            opts["gpu_mem_limit"] = GPU_MEM_MO * 1024 * 1024
        return [("CUDAExecutionProvider", opts), "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


PIECES_ID = {c: i + 1 for i, c in enumerate("PNBRQK")}


def enc_fen(fen):
    f = fen.split(" ")
    noir = f[1] == "b"
    out = bytearray(69)
    r = c = 0
    for ch in f[0]:
        if ch == "/":
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            nous = ch.isupper() != noir
            out[(r if noir else 7 - r) * 8 + c] = PIECES_ID[ch.upper()] + (0 if nous else 6)
            c += 1
    ours, theirs = ("kq", "KQ") if noir else ("KQ", "kq")
    out[64] = ours[0] in f[2]
    out[65] = ours[1] in f[2]
    out[66] = theirs[0] in f[2]
    out[67] = theirs[1] in f[2]
    if f[3] != "-":
        out[68] = ord(f[3][0]) - 96
    return bytes(out)


class Evaluateur:
    """un réseau ONNX : liste de FEN -> centipions du point de vue du camp au trait"""

    def __init__(self, nom, gpu=True, threads=0):
        self.nom = nom
        self.so = ort.SessionOptions()
        if threads:
            self.so.intra_op_num_threads = threads
            self.so.inter_op_num_threads = 1
        global _CUDA_OK
        prov = providers(gpu)
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), self.so, providers=prov)
        self.sur_gpu = "CUDAExecutionProvider" in self.session.get_providers()
        if prov[0] != "CPUExecutionProvider":
            _CUDA_OK = self.sur_gpu
            if not _CUDA_OK:
                print("⚠️ GPU indisponible pour onnxruntime : on continue sur CPU (plus lent mais résultats identiques)")
                ort.set_default_logger_severity(4)
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        if self.sur_gpu and GPU_MEM_MO:
            lot = min(lot, 512)  # petits lots quand la mémoire GPU est plafonnée
        res = []
        for i in range(0, len(fens), lot):
            x = np.frombuffer(b"".join(map(enc_fen, fens[i:i + lot])), np.uint8).reshape(-1, 69).copy()
            n = len(x)
            if self.sur_gpu and GPU_MEM_MO:
                # taille de lot arrondie à une puissance de 2 : peu de formes différentes, donc
                # onnxruntime réutilise ses blocs mémoire au lieu d'en allouer de nouveaux à chaque coup
                taille = max(64, 1 << (n - 1).bit_length())
                x = np.concatenate([x, np.zeros((taille - n, 69), np.uint8)]) if taille > n else x
            try:
                res.append(self.session.run(None, {"board": x})[0][:n])
            except Exception as e:
                if not self.sur_gpu:
                    raise
                # GPU plein : ce réseau passe sur CPU pour la suite (mêmes résultats, plus lent)
                print(f"⚠️ {self.nom} : erreur GPU ({str(e)[:80]}...) -> CPU")
                self.session = ort.InferenceSession(poids(RESEAUX[self.nom]), self.so, providers=["CPUExecutionProvider"])
                self.sur_gpu = False
                res.append(self.session.run(None, {"board": x})[0])
        self.nb_evals += len(fens)
        return np.concatenate(res)

    def eval1(self, fen):
        # l'alpha-bêta réévalue souvent les mêmes positions en quiescence : le réseau est
        # déterministe, donc un cache ne change rien au résultat, juste la vitesse
        v = self.cache.get(fen)
        if v is None:
            if len(self.cache) > 2_000_000:
                self.cache.clear()
            v = self.cache[fen] = float(self.evals([fen])[0])
        return v


# -- P.O.W.N. (NNUE HalfKAv2_hm) : autre encodage d'entrée, même interface -------------

def _bucket_pown(roi, miroir):
    f = chess.square_file(roi)
    if miroir:
        f = 7 - f
    return chess.square_rank(roi) * 4 + f


def features_pown(board, couleur):
    """doit rester identique à features() de Pown/commun_pown.py (vérifié par test)"""
    roi = board.king(couleur)
    miroir = chess.square_file(roi) >= 4
    b = _bucket_pown(roi, miroir)
    idx = []
    for case, piece in board.piece_map().items():
        if piece.piece_type == chess.KING and piece.color == couleur:
            continue
        f, r = chess.square_file(case), chess.square_rank(case)
        if miroir:
            f = 7 - f
        nous = piece.color == couleur
        t = (piece.piece_type - 1) if nous else 5 + (piece.piece_type - 1)
        idx.append(b * 704 + (r * 8 + f) * 11 + t)
    return idx


class EvaluateurPOWN(Evaluateur):
    """P.O.W.N. : ONNX à 4 entrées (index + décalages des deux points de vue) -> logit ; cp = 400 x logit"""

    def __init__(self, nom, gpu=False, threads=0):
        self.nom = nom
        self.so = ort.SessionOptions()
        if threads:
            self.so.intra_op_num_threads = threads
            self.so.inter_op_num_threads = 1
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), self.so, providers=["CPUExecutionProvider"])
        self.sur_gpu = False
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        res = []
        for i in range(0, len(fens), lot):
            idx_n, dec_n, idx_e, dec_e = [], [], [], []
            for fen in fens[i:i + lot]:
                b = chess.Board(fen)
                n, e = features_pown(b, b.turn), features_pown(b, not b.turn)
                dec_n.append(len(idx_n)); idx_n += n
                dec_e.append(len(idx_e)); idx_e += e
            sortie = self.session.run(None, {
                "idx_nous": np.array(idx_n, dtype=np.int64), "dec_nous": np.array(dec_n, dtype=np.int64),
                "idx_eux": np.array(idx_e, dtype=np.int64), "dec_eux": np.array(dec_e, dtype=np.int64)})[0]
            res.append(400.0 * sortie)
        self.nb_evals += len(fens)
        return np.concatenate(res)


# -- recherche « arbre complet » (PAWN, PAWN big, contrôle, SPAWN) -----------------

def arbre(board, depth, ply, feuilles):
    if not any(board.legal_moves):
        return ("t", -(MATE - ply) if board.is_check() else 0)
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return ("t", 0)
    if depth == 0:
        feuilles.append(board.fen())
        return ("f", len(feuilles) - 1)
    fils = []
    for mv in list(board.legal_moves):
        board.push(mv)
        fils.append((mv, arbre(board, depth - 1, ply + 1, feuilles)))
        board.pop()
    return ("n", fils)


def valeur(noeud, v):
    genre, x = noeud
    if genre == "t":
        return x
    if genre == "f":
        return v[x]
    return max(-valeur(c, v) for _, c in x)


def choisir_arbre(board, ev, depth):
    feuilles = []
    racine = arbre(board, depth, 0, feuilles)
    v = ev.evals(feuilles) if feuilles else []
    notes = [(-valeur(c, v), mv) for mv, c in racine[1]]
    return max(notes, key=lambda t: t[0])[1]


# -- recherche alpha-bêta + quiescence (PWN) --------------------------------------

VAL_PIECE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}


def trier_coups(board, coups, priorite):
    def cle(mv):
        if mv == priorite:
            return 10_000
        if board.is_capture(mv):
            prise = board.piece_type_at(mv.to_square) or chess.PAWN
            attaquant = board.piece_type_at(mv.from_square)
            return 1000 + VAL_PIECE[prise] * 10 - VAL_PIECE[attaquant]
        if board.gives_check(mv):
            return 500
        return 0
    return sorted(coups, key=cle, reverse=True)


def quiescence(board, ev, alpha, beta, prof_q):
    stand_pat = ev.eval1(board.fen())
    if stand_pat >= beta:
        return beta
    alpha = max(alpha, stand_pat)
    if prof_q <= 0:
        return alpha
    captures = [m for m in board.legal_moves if board.is_capture(m)]
    for mv in trier_coups(board, captures, None):
        board.push(mv)
        score = -quiescence(board, ev, -beta, -alpha, prof_q - 1)
        board.pop()
        if score >= beta:
            return beta
        alpha = max(alpha, score)
    return alpha


def negamax(board, ev, profondeur, alpha, beta, ply, prof_q, tt, coup_tt):
    alpha0 = alpha
    if not any(board.legal_moves):
        return -(MATE - ply) if board.is_check() else 0
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return 0
    hash_ = chess.polyglot.zobrist_hash(board)
    entree = tt.get(hash_)
    if entree and entree[0] >= profondeur:
        d, v, drapeau, _ = entree
        if drapeau == "exact":
            return v
        if drapeau == "min":
            alpha = max(alpha, v)
        elif drapeau == "max":
            beta = min(beta, v)
        if alpha >= beta:
            return v
    if profondeur <= 0:
        return quiescence(board, ev, alpha, beta, prof_q)
    meilleur, meilleur_coup = -MATE - 1, None
    for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
        board.push(mv)
        score = -negamax(board, ev, profondeur - 1, -beta, -alpha, ply + 1, prof_q, tt, coup_tt)
        board.pop()
        if score > meilleur:
            meilleur, meilleur_coup = score, mv
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
    tt[hash_] = (profondeur, meilleur, drapeau, meilleur_coup)
    if meilleur_coup is not None:
        coup_tt[hash_] = meilleur_coup
    return meilleur


def choisir_alphabeta(board, ev, depth, prof_q=4):
    tt, coup_tt = {}, {}
    meilleur_coup = None
    for d in range(1, depth + 1):
        alpha, beta = -MATE - 1, MATE + 1
        hash_ = chess.polyglot.zobrist_hash(board)
        meilleur_score, meilleur_du_tour = -MATE - 1, None
        for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
            board.push(mv)
            score = -negamax(board, ev, d - 1, -beta, -alpha, 1, prof_q, tt, coup_tt)
            board.pop()
            if score > meilleur_score:
                meilleur_score, meilleur_du_tour = score, mv
            alpha = max(alpha, score)
        meilleur_coup = meilleur_du_tour
        coup_tt[hash_] = meilleur_coup
    return meilleur_coup


# -- recherche « équilibrée » : alpha-bêta améliorée + durée fixée par coup (POWN@t5 = 5 s par coup) ----

class _Temps(Exception):
    pass


class Recherche:
    """Alpha-bêta avec approfondissement itératif jusqu'à épuisement du temps. Par rapport à choisir_alphabeta :
    tri des coups (table, captures MVV-LVA, tueurs, historique), quiescence qui écarte les mauvaises captures,
    extension d'échec, coup nul, réductions (LMR). Le coup rendu est celui de la dernière profondeur terminée."""

    def __init__(self, ev, secondes, prof_max=20, prof_q=4):
        self.ev, self.secondes, self.prof_max, self.prof_q = ev, secondes, prof_max, prof_q
        self.tt, self.tueurs, self.histo, self.noeuds = {}, {}, {}, 0
        self.fin = 0.0
        self.prof_atteinte = 0

    def _horloge(self):
        self.noeuds += 1
        if self.noeuds & 15 == 0 and time.perf_counter() > self.fin:
            raise _Temps()

    def _cle(self, board, mv, coup_tt, ply):
        if mv == coup_tt:
            return 1_000_000
        if board.is_capture(mv) or mv.promotion:
            victime = VAL_PIECE[board.piece_type_at(mv.to_square) or chess.PAWN]
            return 100_000 + victime * 100 - VAL_PIECE[board.piece_type_at(mv.from_square)] + (50 if mv.promotion else 0)
        if mv in self.tueurs.get(ply, ()):
            return 90_000
        return self.histo.get((board.turn, mv.from_square, mv.to_square), 0)

    def quiescence(self, board, alpha, beta, prof):
        self._horloge()
        en_echec = board.is_check()
        if en_echec:
            coups = list(board.legal_moves)
            if not coups:
                return -MATE
            gain = -MATE
        else:
            gain = self.ev.eval1(board.fen())
            if gain >= beta:
                return gain
            alpha = max(alpha, gain)
            if prof <= 0:
                return gain
            coups = []
            for m in board.legal_moves:
                if board.is_capture(m) or m.promotion:
                    victime = VAL_PIECE[board.piece_type_at(m.to_square) or chess.PAWN]
                    # delta : même en gagnant la pièce, on reste loin de alpha
                    if gain + 100 * victime + 200 < alpha and not m.promotion:
                        continue
                    # mauvaise capture : pièce plus chère qui prend une pièce moins chère protégée
                    if (VAL_PIECE[board.piece_type_at(m.from_square)] > victime and not m.promotion
                            and board.is_attacked_by(not board.turn, m.to_square)):
                        continue
                    coups.append(m)
        coups.sort(key=lambda m: self._cle(board, m, None, 0), reverse=True)
        for mv in coups:
            board.push(mv)
            v = -self.quiescence(board, -beta, -alpha, prof - 1)
            board.pop()
            if v > gain:
                gain = v
                if v > alpha:
                    alpha = v
                    if alpha >= beta:
                        break
        return gain

    def negamax(self, board, depth, alpha, beta, ply, nul_ok=True):
        self._horloge()
        coups = list(board.legal_moves)
        if not coups:
            return -(MATE - ply) if board.is_check() else 0
        if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
            return 0
        alpha0 = alpha
        h = chess.polyglot.zobrist_hash(board)
        e = self.tt.get(h)
        coup_tt = e[3] if e else None
        if e and e[0] >= depth and e[2] != "none":
            _, v, drapeau, _ = e
            if drapeau == "exact":
                return v
            if drapeau == "min":
                alpha = max(alpha, v)
            else:
                beta = min(beta, v)
            if alpha >= beta:
                return v
        en_echec = board.is_check()
        if en_echec and ply < 40:
            depth += 1                       # extension d'échec
        if depth <= 0:
            return self.quiescence(board, alpha, beta, self.prof_q)
        # coup nul : si même en passant son tour on reste >= beta, on coupe (pas en finale de pions : zugzwang)
        if (nul_ok and depth >= 3 and not en_echec and beta < MATE - 1000
                and any(board.pieces(t, board.turn) for t in (chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN))):
            board.push(chess.Move.null())
            v = -self.negamax(board, depth - 3, -beta, -beta + 1, ply + 1, False)
            board.pop()
            if v >= beta:
                return beta
        coups.sort(key=lambda m: self._cle(board, m, coup_tt, ply), reverse=True)
        meilleur, meilleur_coup = -MATE - 1, None
        for i, mv in enumerate(coups):
            calme = not board.is_capture(mv) and not mv.promotion
            donne_echec = board.gives_check(mv)
            board.push(mv)
            if i >= 4 and depth >= 3 and calme and not en_echec and not donne_echec:
                v = -self.negamax(board, depth - 2, -alpha - 1, -alpha, ply + 1)      # réduit, fenêtre nulle
                if v > alpha:
                    v = -self.negamax(board, depth - 1, -beta, -alpha, ply + 1)
            else:
                v = -self.negamax(board, depth - 1, -beta, -alpha, ply + 1)
            board.pop()
            if v > meilleur:
                meilleur, meilleur_coup = v, mv
            if v > alpha:
                alpha = v
            if alpha >= beta:
                if calme:
                    t = self.tueurs.setdefault(ply, [])
                    if mv not in t:
                        t.insert(0, mv)
                        del t[2:]
                    k = (board.turn, mv.from_square, mv.to_square)
                    self.histo[k] = self.histo.get(k, 0) + depth * depth
                break
        if abs(meilleur) > MATE - 1000:
            drapeau = "none"                 # score de mat dépendant de la distance : on ne le réutilise pas
        else:
            drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
        self.tt[h] = (depth, meilleur, drapeau, meilleur_coup)
        return meilleur

    def choisir(self, board):
        b = board.copy(stack=True)
        coups = list(b.legal_moves)
        if len(coups) == 1:
            return coups[0]
        debut = time.perf_counter()
        self.fin = debut + self.secondes
        meilleur_coup, ordre = coups[0], coups
        for d in range(1, self.prof_max + 1):
            try:
                alpha, beta, meilleur_score, du_tour = -MATE - 1, MATE + 1, -MATE - 1, None
                for mv in ordre:
                    b.push(mv)
                    v = -self.negamax(b, d - 1, -beta, -alpha, 1)
                    b.pop()
                    if v > meilleur_score:
                        meilleur_score, du_tour = v, mv
                    alpha = max(alpha, v)
            except _Temps:
                break
            meilleur_coup, self.prof_atteinte = du_tour, d
            ordre = [du_tour] + [m for m in ordre if m != du_tour]    # le meilleur d'abord à la profondeur suivante
            if abs(meilleur_score) > MATE - 1000:
                break                                                # mat trouvé : inutile d'aller plus loin
            if time.perf_counter() - debut > 0.4 * self.secondes:
                break                                                # la profondeur suivante coûterait plus que ce qui reste
        return meilleur_coup


DERNIERE = {}   # profondeur atteinte et nombre de noeuds de la dernière recherche (pour les statistiques)


def choisir_temps(board, ev, secondes):
    r = Recherche(ev, secondes)
    mv = r.choisir(board)
    DERNIERE.update(profondeur=r.prof_atteinte, noeuds=r.noeuds)
    return mv


# -- joueurs ------------------------------------------------------------------------
# un « joueur » = un réseau + un algorithme de recherche + une profondeur.
# ex : "PAWN_BIG@arbre2", "SPAWN@arbre1", "PWN@ab3" (PWN = poids SPAWN + alpha-bêta).

def decoder(joueur):
    nom, rech = joueur.split("@")
    if nom == "PWN":
        nom = "SPAWN"
    if rech.startswith("arbre"):
        return nom, "arbre", int(rech[5:])
    if rech.startswith("ab"):
        return nom, "ab", int(rech[2:])
    if rech.startswith("t"):          # ex. "POWN@t5" : recherche équilibrée, 5 secondes par coup
        return nom, "temps", float(rech[1:])
    raise ValueError(joueur)


_EVALS = {}
_SF = {}
GPU = True
THREADS = 0
SF_CHEMIN = None
SF_TEMPS = 0.1


def evaluateur(nom, gpu=None):
    gpu = GPU if gpu is None else gpu
    if (nom, gpu) not in _EVALS:
        classe = EvaluateurPOWN if nom == "POWN" else Evaluateur
        _EVALS[(nom, gpu)] = classe(nom, gpu=gpu, threads=THREADS)
    return _EVALS[(nom, gpu)]


def configurer(gpu=True, threads=0, sf_chemin=None, sf_temps=0.1):
    """à appeler dans chaque processus fils avant de jouer"""
    global GPU, THREADS, SF_CHEMIN, SF_TEMPS
    GPU, THREADS, SF_CHEMIN, SF_TEMPS = gpu, threads, sf_chemin, sf_temps
    _EVALS.clear()
    _SF.clear()


def stockfish(cfg):
    """cfg = "elo1500" (UCI_LimitStrength) ou "skill0" (Skill Level)"""
    if cfg not in _SF:
        e = chess.engine.SimpleEngine.popen_uci(SF_CHEMIN)
        opts = {"Threads": 1, "Hash": 16}
        if cfg.startswith("elo"):
            opts.update({"UCI_LimitStrength": True, "UCI_Elo": int(cfg[3:])})
        elif cfg.startswith("skill"):
            opts["Skill Level"] = int(cfg[5:])
        e.configure(opts)
        _SF[cfg] = e
    return _SF[cfg]


def fermer_stockfish():
    for e in _SF.values():
        e.quit()
    _SF.clear()


def coup(joueur, board):
    if joueur.startswith("SF@"):
        return stockfish(joueur[3:]).play(board, chess.engine.Limit(time=SF_TEMPS)).move
    nom, rech, depth = decoder(joueur)
    b = board.copy(stack=True)
    if rech == "arbre":
        return choisir_arbre(b, evaluateur(nom), depth)
    if rech == "temps":
        return choisir_temps(b, evaluateur(nom, gpu=False), depth)
    # l'alpha-bêta évalue une position à la fois : sur Kaggle la latence CPU (~6 ms)
    # est deux fois plus basse que celle du GPU (~14 ms), donc on reste sur CPU
    return choisir_alphabeta(b, evaluateur(nom, gpu=False), depth)


# -- Stockfish -------------------------------------------------------------------------

URLS_STOCKFISH = [
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17.1/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_16.1/stockfish-ubuntu-x86-64-avx2.tar",
]


def rendre_executable(binaire):
    """/kaggle/working interdit d'exécuter des programmes : on copie Stockfish ailleurs
    et on garde le premier emplacement où il démarre vraiment"""
    for dest in ("/usr/local/bin/stockfish-pawn", "/tmp/stockfish-pawn", binaire):
        try:
            if dest != binaire:
                shutil.copy(binaire, dest)
            os.chmod(dest, 0o755)
            subprocess.run([dest, "quit"], check=True, timeout=10, capture_output=True)
            return dest
        except Exception as e:
            print("Stockfish ne démarre pas depuis", dest, ":", e)
    raise RuntimeError("Stockfish ne peut être exécuté nulle part")


def installer_stockfish(dossier=None):
    dossier = dossier or os.path.join(os.path.dirname(os.path.abspath(DOSSIER_POIDS)), "stockfish")
    deja = glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
    deja = [d for d in deja if os.path.isfile(d) and not d.endswith(".tar")]
    if deja:
        return rendre_executable(deja[0])
    os.makedirs(dossier, exist_ok=True)
    for url in URLS_STOCKFISH:
        try:
            tar = os.path.join(dossier, "sf.tar")
            urllib.request.urlretrieve(url, tar)
            with tarfile.open(tar) as t:
                t.extractall(dossier)
            binaire = [d for d in glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
                       if os.path.isfile(d) and not d.endswith(".tar")][0]
            return rendre_executable(binaire)
        except Exception as e:
            print("échec", url, e)
    if shutil.which("stockfish") is None:
        subprocess.run("apt-get -qq install -y stockfish", shell=True)
    for p in (shutil.which("stockfish"), "/usr/games/stockfish"):
        if p and os.path.exists(p):
            return p
    raise RuntimeError("impossible d'installer Stockfish")


# -- puzzles Lichess ---------------------------------------------------------------

def resoudre_puzzle(joueur, fen, moves):
    """format Lichess : FEN avant le coup adverse, moves[0] = coup adverse, puis solution.
    Comme sur Lichess, un mat est toujours accepté même si ce n'est pas le coup attendu.
    renvoie (résolu, premier_coup_bon, nb_coups_joués, secondes)"""
    b = chess.Board(fen)
    moves = moves.split()
    b.push_uci(moves[0])
    premier, joues, t0 = None, 0, time.perf_counter()
    for i in range(1, len(moves), 2):
        attendu = chess.Move.from_uci(moves[i])
        mv = coup(joueur, b)
        joues += 1
        b.push(mv)
        mat = b.is_checkmate()
        b.pop()
        ok = mv == attendu or mat
        if premier is None:
            premier = ok
        if not ok:
            return False, premier, joues, time.perf_counter() - t0
        if mat:
            break
        b.push(attendu)
        if i + 1 < len(moves):
            b.push_uci(moves[i + 1])
    return True, premier, joues, time.perf_counter() - t0


# -- parties -------------------------------------------------------------------------

def ecrire_suivi(suivi, blancs, noirs, b, ouverture, debut):
    # position en cours dans un petit fichier JSON, lu par le notebook pour l'afficher en direct
    dernier = b.peek() if b.move_stack else None
    san = ""
    if dernier:
        prec = b.copy()
        prec.pop()
        san = prec.san(dernier)
    tmp = suivi + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"blancs": blancs, "noirs": noirs, "fen": b.fen(), "dernier": dernier.uci() if dernier else "",
                   "san": san, "ply": b.ply(), "ouverture": ouverture, "debut": debut}, f)
    os.replace(tmp, suivi)


def jouer_partie(blancs, noirs, ouverture, max_plies=300, suivi=None, nom_ouverture=""):
    """ouverture = liste de coups UCI joués d'office ; renvoie un dict (résultat, pgn, temps)"""
    import chess.pgn
    b = chess.Board()
    positions = []  # chaque coup : qui, coup, position obtenue, temps de réflexion
    for u in ouverture:
        mv = chess.Move.from_uci(u)
        positions.append({"ply": b.ply() + 1, "joueur": "ouverture", "san": b.san(mv), "uci": u, "secondes": 0.0})
        b.push(mv)
        positions[-1]["fen"] = b.fen()
    temps = {blancs: 0.0, noirs: 0.0}
    nb = {blancs: 0, noirs: 0}
    debut = time.time()
    if suivi:
        ecrire_suivi(suivi, blancs, noirs, b, nom_ouverture, debut)
    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
        j = blancs if b.turn == chess.WHITE else noirs
        t0 = time.perf_counter()
        mv = coup(j, b)
        dt = time.perf_counter() - t0
        temps[j] += dt
        nb[j] += 1
        positions.append({"ply": b.ply() + 1, "joueur": j, "san": b.san(mv), "uci": mv.uci(), "secondes": round(dt, 3)})
        b.push(mv)
        positions[-1]["fen"] = b.fen()
        if suivi:
            ecrire_suivi(suivi, blancs, noirs, b, nom_ouverture, debut)
    res = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
    fin = b.outcome(claim_draw=True)
    jeu = chess.pgn.Game.from_board(b)
    jeu.headers.update(Event="Benchmark P.A.W.N.", White=blancs, Black=noirs, Result=res)
    return {
        "blancs": blancs, "noirs": noirs, "resultat": res,
        "fin": fin.termination.name if fin else "MAX_PLIES",
        "plies": b.ply(), "pgn": str(jeu),
        "s_par_coup_blancs": temps[blancs] / max(1, nb[blancs]),
        "s_par_coup_noirs": temps[noirs] / max(1, nb[noirs]),
        "positions": positions,
    }


# -- tâches pour ProcessPoolExecutor (doivent être importables depuis ce module) -------

def nb_gpu():
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        return max(1, sum(l.startswith("GPU") for l in r.stdout.splitlines()))
    except Exception:
        return 1


def init_worker(gpu, threads, sf_chemin=None, sf_temps=0.1, gpu_mem_mo=None):
    # plusieurs processus x plusieurs réseaux sur un GPU : sans plafond, onnxruntime réserve
    # la mémoire trop largement et le T4 sature (BFCArena / CUBLAS_STATUS_ALLOC_FAILED).
    # Les processus sont répartis sur les GPU disponibles, avec un plafond par réseau.
    global GPU_MEM_MO, GPU_ID
    n = nb_gpu()
    GPU_ID = os.getpid() % n
    GPU_MEM_MO = gpu_mem_mo or (700 if n == 1 else 1200)
    configurer(gpu=gpu, threads=threads, sf_chemin=sf_chemin, sf_temps=sf_temps)


def tache_puzzle(args):
    joueur, pid, fen, moves = args
    ok, premier, joues, sec = resoudre_puzzle(joueur, fen, moves)
    return {"joueur": joueur, "PuzzleId": pid, "resolu": ok, "premier_coup": premier,
            "coups_joues": joues, "secondes": sec}


def tache_partie(args):
    blancs, noirs, ouverture, max_plies, id_ouv = args[:5]
    suivi = args[5] if len(args) > 5 else None   # dossier de suivi en direct (facultatif)
    fichier = None
    if suivi:
        os.makedirs(suivi, exist_ok=True)
        fichier = os.path.join(suivi, f"{blancs}_{noirs}_{id_ouv}.json".replace(" ", "-").replace("@", "-"))
    try:
        r = jouer_partie(blancs, noirs, ouverture, max_plies, fichier, id_ouv)
    finally:
        # un Stockfish ouvert garde un thread vivant qui empêche le processus fils de se terminer
        fermer_stockfish()
        if fichier and os.path.exists(fichier):
            os.remove(fichier)
    r["ouverture"] = id_ouv
    return r

In [ ]:
import os, sys, time, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import chess
sys.path.insert(0, os.getcwd())  # pour que les processus fils trouvent commun_echecs.py
import onnxruntime as ort
import commun_echecs as ce

print("onnxruntime", ort.__version__, "| providers dispo :", ort.get_available_providers())
ev_test = ce.Evaluateur("PAWN")
print("session PAWN sur :", ev_test.session.get_providers())
print("éval position initiale :", ev_test.eval1(chess.STARTING_FEN), "cp")

In [ ]:
try:
    print("poids P.O.W.N. trouvés :", ce.poids("Pown/pown.onnx"))
except Exception as e:
    raise SystemExit("❌ pown.onnx introuvable : crée un Dataset Kaggle avec ce fichier et ajoute-le en Input. (" + str(e)[:100] + ")")

In [ ]:
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

POSITIONS = {
    "départ": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "italienne": "r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "milieu calme": "r2q1rk1/ppp2ppp/2np1n2/2b1p1B1/2B1P1b1/2NP1N2/PPP2PPP/R2Q1RK1 w - - 0 8",
    "kiwipete (tactique)": "r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1",
    "finale de tours": "8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 1",
    "milieu fermé": "2r2rk1/1p2qppp/p1n1pn2/3p4/3P4/P1NBPN2/1P3PPP/2RQ1RK1 w - - 0 1",
    "pions + roi": "8/8/4k3/3p4/3P4/4K3/8/8 w - - 0 1",
    "attaque": "r1b2rk1/pp1nqppp/2p1pn2/3p4/2PP4/2NBPN2/PP3PPP/R2QK2R w KQ - 0 9",
}
DEPTHS = [1, 2, 3, 4]   # on s'arrête dès qu'une profondeur dépasse LIMITE_S par coup
LIMITE_S = 20
ev = ce.evaluateur("POWN", gpu=False)
lignes, choisis = [], {}
for d in DEPTHS:
    tt, ne, choisis[d] = 0.0, 0, {}
    for nom, fen in POSITIONS.items():
        ev.cache.clear()
        n0 = ev.nb_evals
        t = time.perf_counter()
        mv = ce.choisir_alphabeta(chess.Board(fen), ev, d)
        tt += time.perf_counter() - t
        ne += ev.nb_evals - n0
        choisis[d][nom] = mv.uci()
    lignes.append({"profondeur": d, "s / coup": tt / len(POSITIONS), "positions évaluées / coup": ne / len(POSITIONS),
                   "évaluations / s": ne / tt})
    print(f"ab{d} : {tt / len(POSITIONS):.2f} s/coup, {ne / len(POSITIONS):,.0f} positions évaluées/coup, {ne / tt:,.0f} évaluations/s")
    if tt / len(POSITIONS) > LIMITE_S:
        print(f"plus de {LIMITE_S} s par coup : trop lent pour continuer, on s'arrête là"); break
vitesse = pd.DataFrame(lignes).set_index("profondeur")
prof_max = max(choisis)
vitesse["même coup que la profondeur max (%)"] = [100 * np.mean([choisis[d][n] == choisis[prof_max][n] for n in POSITIONS]) for d in vitesse.index]
vitesse.to_csv(f"{SORTIE}/pown_ab_vitesse.csv")
vitesse.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
vitesse["s / coup"].plot.bar(ax=ax[0], color="#58a", logy=True, rot=0, title="secondes par coup (échelle log)")
vitesse["évaluations / s"].plot.bar(ax=ax[1], color="#a85", rot=0, title="évaluations par seconde")
plt.tight_layout(); plt.savefig(f"{SORTIE}/pown_ab_vitesse.png", dpi=120); plt.show()

In [ ]:
# recherche « équilibrée » : durée fixée par coup, la profondeur s'adapte à la position
lignes = []
for T in [1, 3, 5]:
    profs, noeuds, evs, coups_t = [], 0, 0, {}
    for nom, fen in POSITIONS.items():
        ev.cache.clear()
        n0 = ev.nb_evals
        t = time.perf_counter()
        mv = ce.choisir_temps(chess.Board(fen), ev, T)
        profs.append(ce.DERNIERE["profondeur"]); noeuds += ce.DERNIERE["noeuds"]; evs += ev.nb_evals - n0
    lignes.append({"secondes / coup": T, "profondeur atteinte (moyenne)": np.mean(profs), "profondeur min": min(profs),
                   "profondeur max": max(profs), "noeuds / coup": noeuds / len(POSITIONS), "positions évaluées / coup": evs / len(POSITIONS)})
    print(f"{T} s/coup : profondeur {np.mean(profs):.1f} en moyenne (de {min(profs)} à {max(profs)})")
temps_tab = pd.DataFrame(lignes).set_index("secondes / coup")
temps_tab.to_csv(f"{SORTIE}/pown_recherche_temps.csv")
temps_tab.round(1)

In [ ]:
# ---- réglages ----
TRANCHES = list(range(400, 2801, 200))   # bornes des tranches de classement puzzle
PAR_TRANCHE = 25                         # puzzles par tranche (12 tranches -> 1200 puzzles)
PAR_TRANCHE_LENT = 40                     # pour les joueurs lents (PWN@ab3)
JOUEURS = ["POWN@ab2", "POWN@t2", "PWN@t2"]
LENTS = set()
GPU = False
NB_WORKERS = os.cpu_count()
THREADS = 1                                # threads onnxruntime par processus (CPU)
SEED = 0
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

## 1. Base de puzzles Lichess

In [ ]:
import io, urllib.request
FICHIER = f"{SORTIE}/lichess_db_puzzle.csv.zst"
COLS = ["PuzzleId", "FEN", "Moves", "Rating", "RatingDeviation", "Popularity", "NbPlays", "Themes"]
try:
    import zstandard
    if not os.path.exists(FICHIER):
        urllib.request.urlretrieve("https://database.lichess.org/lichess_db_puzzle.csv.zst", FICHIER)
    with open(FICHIER, "rb") as fh:
        texte = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(fh), encoding="utf-8")
        puzzles = pd.read_csv(texte, usecols=COLS)
except Exception as e:
    print("database.lichess.org indisponible :", e, "-> Hugging Face")
    from datasets import load_dataset
    puzzles = load_dataset("Lichess/chess-puzzles", split="train").to_pandas()[COLS]
print(f"{len(puzzles):,} puzzles")

In [ ]:
# puzzles fiables : classement stable, bien notés, beaucoup joués
ok = puzzles[(puzzles.RatingDeviation < 90) & (puzzles.Popularity > 80) & (puzzles.NbPlays > 1000)].copy()
ok["tranche"] = pd.cut(ok.Rating, TRANCHES, right=False)
ok = ok.dropna(subset=["tranche"])
ok = ok.sample(frac=1, random_state=SEED)  # mélange, puis les PAR_TRANCHE premiers de chaque tranche
echant = ok[ok.groupby("tranche", observed=True).cumcount() < PAR_TRANCHE].sort_values("Rating").reset_index(drop=True)
# sous-échantillon pour les joueurs lents : les PAR_TRANCHE_LENT premiers de chaque tranche
echant["lent"] = echant.groupby("tranche", observed=True).cumcount() < PAR_TRANCHE_LENT
echant.to_csv(f"{SORTIE}/bench2_puzzles_echantillon.csv", index=False)
print(len(echant), "puzzles,", echant.lent.sum(), "pour les joueurs lents")
echant.groupby("tranche", observed=True).size()

## 2. Résolution (en parallèle, avec reprise)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

RESULTATS = f"{SORTIE}/bench2_resultats_bruts.csv"
deja = pd.read_csv(RESULTATS) if os.path.exists(RESULTATS) else pd.DataFrame(columns=["joueur", "PuzzleId"])
fait = set(zip(deja.joueur, deja.PuzzleId))

for joueur in JOUEURS:
    lot = echant[echant.lent] if joueur in LENTS else echant
    taches = [(joueur, r.PuzzleId, r.FEN, r.Moves) for r in lot.itertuples() if (joueur, r.PuzzleId) not in fait]
    if not taches:
        print(joueur, ": déjà fait"); continue
    t0 = time.time()
    with ProcessPoolExecutor(NB_WORKERS, mp_context=mp.get_context("spawn"),
                             initializer=ce.init_worker, initargs=(GPU, THREADS)) as ex:
        tampon = []
        for r in tqdm(ex.map(ce.tache_puzzle, taches, chunksize=2), total=len(taches), desc=joueur):
            tampon.append(r)
            if len(tampon) >= 50:
                pd.DataFrame(tampon).to_csv(RESULTATS, mode="a", header=not os.path.exists(RESULTATS), index=False)
                tampon = []
        if tampon:
            pd.DataFrame(tampon).to_csv(RESULTATS, mode="a", header=not os.path.exists(RESULTATS), index=False)
    print(f"{joueur} : {len(taches)} puzzles en {(time.time() - t0) / 60:.1f} min")

## 3. Résultats

In [ ]:
from scipy.optimize import minimize_scalar

res = pd.read_csv(RESULTATS).drop_duplicates(["joueur", "PuzzleId"]).merge(echant, on="PuzzleId")
res["resolu"] = res["resolu"].astype(bool); res["premier_coup"] = res["premier_coup"].astype(bool)

def elo_puzzle(ratings, ok):
    """classement E tel que P(résolu) = 1 / (1 + 10^((R - E) / 400)) colle le mieux aux résultats"""
    r, y = np.asarray(ratings, float), np.asarray(ok, float)
    def nll(e):
        p = np.clip(1 / (1 + 10 ** ((r - e) / 400)), 1e-9, 1 - 1e-9)
        return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return minimize_scalar(nll, bounds=(-500, 4000), method="bounded").x

def resume(g):
    return pd.Series({
        "puzzles": len(g),
        "résolus (%)": 100 * g.resolu.mean(),
        "premier coup (%)": 100 * g.premier_coup.mean(),
        "Elo puzzle": elo_puzzle(g.Rating, g.resolu),
        "s / coup": g.secondes.sum() / g.coups_joues.sum(),
    })

tableau = res.groupby("joueur").apply(resume).loc[[j for j in JOUEURS if j in set(res.joueur)]]
# même comparaison sur le sous-échantillon commun à tous les joueurs (utile pour PWN@ab3)
tableau["Elo puzzle (sous-éch. commun)"] = res[res.lent].groupby("joueur").apply(lambda g: elo_puzzle(g.Rating, g.resolu))
tableau.to_csv(f"{SORTIE}/bench2_tableau.csv")
tableau.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
par_tranche = res.groupby(["joueur", "tranche"], observed=True).resolu.mean().unstack(0) * 100
par_tranche.index = [str(i.left) if hasattr(i, "left") else str(i).split(",")[0].strip("[") for i in par_tranche.index]
par_tranche.to_csv(f"{SORTIE}/bench2_par_difficulte.csv", index_label="classement puzzle")
par_tranche[[j for j in JOUEURS if j in par_tranche]].plot(ax=axes[0], marker="o")
axes[0].set(xlabel="classement du puzzle", ylabel="résolus (%)", title="Taux de résolution par difficulté")
axes[0].grid(alpha=.3)
tableau["Elo puzzle"].plot.barh(ax=axes[1], color="#58a", title="Elo puzzle estimé")
for i, v in enumerate(tableau["Elo puzzle"]):
    axes[1].text(v, i, f" {v:.0f}", va="center")
plt.tight_layout(); plt.savefig(f"{SORTIE}/bench2_puzzles.png", dpi=120); plt.show()

In [ ]:
THEMES = ["mateIn1", "mateIn2", "mateIn3", "fork", "pin", "skewer", "discoveredAttack", "hangingPiece",
          "sacrifice", "deflection", "attraction", "kingsideAttack", "defensiveMove", "quietMove",
          "opening", "middlegame", "endgame", "pawnEndgame", "rookEndgame", "short", "long", "veryLong"]
lignes = []
for t in THEMES:
    g = res[res.Themes.str.split().apply(lambda l: t in l)]
    if len(g):
        s = g.groupby("joueur").resolu.mean() * 100
        s["n (par joueur)"] = g.groupby("joueur").size().max()
        s.name = t
        lignes.append(s)
themes = pd.DataFrame(lignes)[[j for j in JOUEURS if j in set(res.joueur)] + ["n (par joueur)"]]
themes.to_csv(f"{SORTIE}/bench2_themes.csv")
themes.style.format("{:.0f}").background_gradient(axis=1, cmap="RdYlGn", subset=themes.columns[:-1])

### Tests de McNemar
Sur les mêmes puzzles : combien A résout que B rate (et inversement), et est-ce significatif ?

In [ ]:
from scipy.stats import binomtest
PAIRES = [("POWN@ab2", "POWN@t2"), ("PWN@t2", "POWN@t2")]
piv = res.pivot_table(index="PuzzleId", columns="joueur", values="resolu")
lignes = []
for a, b in PAIRES:
    if a in piv and b in piv:
        x = piv[[a, b]].dropna().astype(bool)
        seul_b, seul_a = int((~x[a] & x[b]).sum()), int((x[a] & ~x[b]).sum())
        p = binomtest(seul_b, seul_a + seul_b).pvalue if seul_a + seul_b else 1.0
        lignes.append({"A": a, "B": b, "puzzles": len(x), "B résout, pas A": seul_b, "A résout, pas B": seul_a,
                       "p-value": p, "significatif (5 %)": p < 0.05})
mcnemar = pd.DataFrame(lignes); mcnemar.to_csv(f"{SORTIE}/bench2_mcnemar.csv", index=False)
mcnemar

### Comment lire les résultats
- **Vitesse** : le temps par coup est multiplié par ~5-10 à chaque profondeur en plus. Si ab3 prend moins d'une seconde, ab4 et ab5 sont jouables.
- **Même coup que la profondeur max** : s'il est bas pour ab1/ab2, la profondeur change beaucoup le jeu (la recherche apporte beaucoup).
- **Puzzles** : compare **POWN@ab2 à PWN@ab2** (même recherche, réseau différent) pour juger le réseau, et **POWN@ab1 → ab2 → ab3** pour juger l'apport de la profondeur.
- Chiffres à ne pas sur-interpréter : ~300 puzzles donnent une marge d'erreur de plusieurs dizaines d'Elo puzzle.